# Lab 0 Task 0.2.1 — Transfer Learning from ImageNet
In this section, we use AlexNet on CIFAR-10 in two settings:
1. Fine-tuning the whole network
2. Feature extraction with pretrained weights

## Setup & Imports

In [1]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import alexnet, AlexNet_Weights
from typing import List, Dict, Tuple, Any
from utils import make_loaders, evaluate, train, validate, fit

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [2]:
LOG_WANDB = False  # Notebook level parameter for enabling/disabling wandb logging
NUM_WORKERS = 8
PIN_MEMORY = True

## Load CIFAR-10

We apply ImageNet normalization and resize all images to 224x224 so they match AlexNet's expected input.

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 256
VAL_FRACTION = 0.2  # 20% of training set used for validation

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)

transform_alexnet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform_alexnet)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform_alexnet)

classes: list[str] = train_set.classes
print("Classes:", classes)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_loader, val_loader, test_loader = make_loaders(
    train_set, test_set,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Dataset split — train: 40,000  val: 10,000  test: 10,000


## AlexNet model
We replace the last classifier layer so the output dimension becomes 10.

In [4]:
def build_alexnet(pretrained=False, freeze_features=False):
    weights = AlexNet_Weights.DEFAULT if pretrained else None
    model = alexnet(weights=weights)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, 10)

    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
        for layer in model.classifier[:-1]:
            for p in layer.parameters():
                p.requires_grad = False

    return model

## Experiment A — Fine-tuning AlexNet on CIFAR-10
Here we train the whole network end-to-end.

In [5]:
NUM_EPOCHS = 2
LEARNING_RATE = 1e-4

model_ft = build_alexnet(pretrained=False, freeze_features=False).to(device)
optimizer_ft = optim.Adam(model_ft.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Fine-tuning",
    tags=["Task 0.2.1", "Fine-tuning"],
    config={"pretrained": False, "freeze_features": False, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_ft = fit(
    model=model_ft,
    optimizer=optimizer_ft,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB,
)

Epoch 1/2 | train loss 1.7490, train acc 35.62% | val loss 1.4659, val acc 46.00%
Epoch 2/2 | train loss 1.3360, train acc 51.74% | val loss 1.2020, val acc 56.16%

Restored best weights (val acc 56.16%)


In [6]:
_, test_acc = evaluate(model=model_ft, test_loader=test_loader,
                       criterion=nn.CrossEntropyLoss(), label="AlexNet Fine-tuning")

[AlexNet Fine-tuning] Test loss: 1.1894 | Test acc: 56.56%


## Experiment B — Feature extraction with pretrained AlexNet
Here we freeze the backbone and train only the final classifier layer.

In [7]:
NUM_EPOCHS = 2
LEARNING_RATE = 1e-3

model_fe = build_alexnet(pretrained=True, freeze_features=True).to(device)
optimizer_fe = optim.Adam([p for p in model_fe.parameters() if p.requires_grad], lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Feature Extraction",
    tags=["Task 0.2.1", "Feature Extraction"],
    config={"pretrained": True, "freeze_features": True, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_fe = fit(
    model=model_fe,
    optimizer=optimizer_fe,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB,
)

Epoch 1/2 | train loss 0.7643, train acc 73.59% | val loss 0.5577, val acc 80.70%
Epoch 2/2 | train loss 0.5993, train acc 78.93% | val loss 0.5313, val acc 81.02%

Restored best weights (val acc 81.02%)


In [8]:
_, test_acc = evaluate(model=model_fe, test_loader=test_loader,
                       criterion=nn.CrossEntropyLoss(), label="AlexNet Feature Extraction")

[AlexNet Feature Extraction] Test loss: 0.5301 | Test acc: 81.38%


## Brief comparison
Fine-tuning updates the whole AlexNet, so the model can adapt to CIFAR-10 more strongly.
Feature extraction keeps pretrained layers frozen, so only the new classifier learns.
That is why fine-tuning usually performs better when the target dataset is different from ImageNet.

In [9]:
print("===== Final Comparison (Accuracy) =====")
print(f"Fine-tuning     best val acc: {max(history_ft['val_acc']):.2f}%")
print(f"Feature extract best val acc: {max(history_fe['val_acc']):.2f}%")

===== Final Comparison (Accuracy) =====
Fine-tuning     best val acc: 56.16%
Feature extract best val acc: 81.02%
